# Train and backtest

Fits a logistic regression with an expanding week-by-week window (no future games leak into a prediction). Reports accuracy against a simple point-differential baseline and shows which features the model leans on.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from nfl_predictor.config import FEATURE_LABELS
from nfl_predictor.models.train import (
    coefficient_table,
    reliability_bins,
    rolling_backtest,
    save_metrics,
    train_final_model,
)
from nfl_predictor.pipeline import build_feature_table

pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

In [ ]:
features = build_feature_table()
print(f"Games in feature table: {len(features):,}")
print(f"Completed games: {features['home_win'].notna().sum():,}")

In [ ]:
backtest, metrics = rolling_backtest(features)
save_metrics(metrics)
pd.Series(metrics)

Accuracy is straight-up winners on held-out weeks. Brier score is closer to 0 when probabilities are well calibrated. The baseline always picks the team with the better prior scoring margin.

In [ ]:
by_season = (
    backtest.assign(
        correct=lambda d: d["pred_home_win"] == d["home_win"],
        sq_err=lambda d: (d["pred_proba"] - d["home_win"]) ** 2,
    )
    .groupby("season")
    .agg(games=("home_win", "size"), accuracy=("correct", "mean"), brier=("sq_err", "mean"))
    .reset_index()
)
display(by_season)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(by_season["season"].astype(int).astype(str), by_season["accuracy"])
ax.axhline(metrics["accuracy"], color="black", linestyle="--", label="overall")
ax.axhline(metrics["home_win_rate"], color="gray", linestyle=":", label="always home")
ax.set_ylim(0.45, 0.75)
ax.set_ylabel("Accuracy")
ax.set_title("Backtest accuracy by season")
ax.legend()
plt.show()

In [ ]:
calib = reliability_bins(backtest)
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], color="gray", linestyle="--")
ax.plot(calib["predicted"], calib["actual"], marker="o")
ax.set_xlabel("Predicted P(home win)")
ax.set_ylabel("Actual home win rate")
ax.set_title("Reliability")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.show()
calib

## What the model uses

Positive coefficients raise P(home win). Injury penalties and pass-heavy-in-bad-weather on the home side should come in negative if those features are working.

In [ ]:
artifact = train_final_model(features, save=True)
coef = coefficient_table(artifact["model"], artifact["feature_columns"])
coef["label"] = coef["feature"].map(FEATURE_LABELS)
display(coef.head(20))

top = coef.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top["label"], top["coefficient"])
ax.axvline(0, color="black", linewidth=1)
ax.set_title("Largest logistic coefficients")
plt.tight_layout()
plt.show()
print(f"Saved model trained on {artifact['n_games']} games, seasons {artifact['seasons']}")